In [ ]:
# Libraries
import numpy as np
import pandas as pd
from scipy import stats

In [15]:
# Master tables.
masterTableHuman = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogs.txt", sep="\t")
masterTableMouse = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogs.txt", sep="\t")

# Add Expression Profile Information

In [16]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

# Merge the paralogue table with the expression profile data. Have to do this twice for the parental and duaghter copies.
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Parental "), on="Gene stable ID", how="left")
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Daughter "), left_on="Human paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Merge the paralogue table with the expression profile data. Have to do this twice for the parental and duaghter copies.
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Parental "), on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Daughter "), left_on="Mouse paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Calculate Distances

In [17]:
def calcTEC(geneOne, geneTwo):
    # Turns the expression profiles into a binary vector that tells us whether a gene is expressed (TPM > 1) or not in a tissue. Turn the dataframe into a series.
    humanOrthoBinary = (geneOne[:-1] > 1)
    mouseOrthoBinary = (geneTwo[:-1] > 1)

    # Finds the number of tissues that are found in one species and not in the other.
    humanOnlyTissueNum = (humanOrthoBinary & ~mouseOrthoBinary).sum()
    mouseOnlyTissueNum = (mouseOrthoBinary & ~humanOrthoBinary).sum()

    # Using the binary vector, we can calculate the total number of tissues the gene is expressed in.
    humanTotalTissue = humanOrthoBinary.sum()
    mouseTotalTissue = mouseOrthoBinary.sum()

    # If any gene is not expressed in any tissue, the TEC formula will output an error. We handle this case by outputting NaN.
    if humanTotalTissue == 0 or mouseTotalTissue == 0:
        return np.nan
    else:
        return ((humanOnlyTissueNum / humanTotalTissue) + (mouseOnlyTissueNum / mouseTotalTissue)) / 2

In [18]:
# Grabs the parental and daughter copy expression profiles.
parentalEPHuman = masterTableHuman.filter(like="Parental")
daughterEPHuman = masterTableHuman.filter(like="Daughter")

# Calculates Euclidean distance.
masterTableHuman["EuclidDist"] = np.linalg.norm(parentalEPHuman.select_dtypes(include="number").to_numpy() - daughterEPHuman.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableHuman["EuclidDistNorm"] = np.linalg.norm(parentalEPHuman.select_dtypes(include="number").div(np.linalg.norm(parentalEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - daughterEPHuman.select_dtypes(include="number").div(np.linalg.norm(daughterEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableHuman["EuclidDistLog"] = np.linalg.norm(np.log2(parentalEPHuman.select_dtypes(include="number") + 1).to_numpy() - np.log2(daughterEPHuman.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename daughterEPHuman's columns to parentalEPHuman'
masterTableHuman["PearDist"] = 1 - parentalEPHuman.select_dtypes(include="number").corrwith(daughterEPHuman.select_dtypes(include="number").rename(columns=dict(zip(daughterEPHuman.columns, parentalEPHuman.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableHuman["TEC"] = [calcTEC(parentCopy, daughterCopy) for parentCopy, daughterCopy in zip(parentalEPHuman.select_dtypes(include="number").to_numpy(), daughterEPHuman.select_dtypes(include="number").to_numpy())]

In [19]:
# Grabs the parental and daughter copy expression profiles.
parentalEPMouse = masterTableMouse.filter(like="Parental")
daughterEPMouse = masterTableMouse.filter(like="Daughter")

# Calculates Euclidean distance.
masterTableMouse["EuclidDist"] = np.linalg.norm(parentalEPMouse.select_dtypes(include="number").to_numpy() - daughterEPMouse.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableMouse["EuclidDistNorm"] = np.linalg.norm(parentalEPMouse.select_dtypes(include="number").div(np.linalg.norm(parentalEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - daughterEPMouse.select_dtypes(include="number").div(np.linalg.norm(daughterEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableMouse["EuclidDistLog"] = np.linalg.norm(np.log2(parentalEPMouse.select_dtypes(include="number") + 1).to_numpy() - np.log2(daughterEPMouse.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename daughterEPMouse's columns to parentalEPMouse'
masterTableMouse["PearDist"] = 1 - parentalEPMouse.select_dtypes(include="number").corrwith(daughterEPMouse.select_dtypes(include="number").rename(columns=dict(zip(daughterEPMouse.columns, parentalEPMouse.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableMouse["TEC"] = [calcTEC(parentCopy, daughterCopy) for parentCopy, daughterCopy in zip(parentalEPMouse.select_dtypes(include="number").to_numpy(), daughterEPMouse.select_dtypes(include="number").to_numpy())]

# Number of Duplicates

In [20]:
humanPairs = masterTableHuman.iloc[:, 0:2]
mousePairs = masterTableMouse.iloc[:, 0:2]

In [21]:
masterTableHuman = masterTableHuman.merge(humanPairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Human paralogue gene stable ID_y": "Number of Duplicates"})
masterTableMouse = masterTableMouse.merge(mousePairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Mouse paralogue gene stable ID_y": "Number of Duplicates"})

# Exons

In [22]:
humanExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanExons.txt", sep="\t")
mouseExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseExons.txt", sep="\t")

humanExonsGrouped = humanExons.groupby("Gene stable ID")["Exon stable ID"].agg(", ".join)
mouseExonsGrouped = mouseExons.groupby("Gene stable ID")["Exon stable ID"].agg(", ".join)

In [23]:
masterTableHuman = masterTableHuman.merge(humanExonsGrouped, on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(mouseExonsGrouped, on="Gene stable ID", how="left")

# Create Master Tables

In [ ]:
# masterTableHuman.to_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.csv", index=False)
# masterTableHuman.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.parquet", index=False)

# masterTableMouse.to_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.csv", index=False)
# masterTableMouse.to_parquet("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.parquet", index=False)

# Correlation between Number of Copies and Distances

In [ ]:
humanMasterCorr = masterTableHuman.loc[:, ["EuclidDist", "EuclidDistNorm", "EuclidDistLog", "PearDist", "TEC"]].corrwith(masterTableHuman["Number of Duplicates"], method="spearman")
mouseMasterCorr = masterTableMouse.loc[:, ["EuclidDist", "EuclidDistNorm", "EuclidDistLog", "PearDist", "TEC"]].corrwith(masterTableMouse["Number of Duplicates"], method="spearman")

display(humanMasterCorr)
display(mouseMasterCorr)

EuclidDist       -0.078683
EuclidDistNorm    0.204842
EuclidDistLog    -0.481140
PearDist          0.147176
TEC               0.024287
dtype: float64

EuclidDist       -0.070956
EuclidDistNorm    0.185681
EuclidDistLog    -0.458570
PearDist          0.150406
TEC               0.039629
dtype: float64